In [6]:
import numpy as np

from torch.utils.data import Dataset
from monai.transforms import *
from monai import transforms as t
from monai.data.dataloader import DataLoader as MonaiDataLoader
from datasets import MultiEpochsDataLoader
import nibabel as nib
import pandas as pd
import numpy as np
import torch

class ADNISequenceDataset(Dataset):
    """
    A dataset class that loads all scans of a given subject as a single data point.
    The scans are ordered by acquisition date to form a temporal sequence.
    Suitable for use in sequence models such as transformers.
    """
    def __init__(self, csv_file, subset='train', aug_p=0.8):
        """
        :param csv_file: Path to the CSV file containing the metadata.
        :param subset: Subset type ('train' or 'test') to determine which transforms to apply.
        :param aug_p: The augmentation probability.
        """
        # Load the CSV into a pandas DataFrame
        self.df = pd.read_csv(csv_file)
        
        # Convert acquisition date to datetime for sorting
        self.df['Acq Date'] = pd.to_datetime(self.df['Acq Date'])
        
        # Group by subject and sort by acquisition date
        self.subject_groups = self.df.groupby('Subject').apply(lambda x: x.sort_values(by='Acq Date'))
        
        self.subjects = list(self.subject_groups.index.levels[0])  # Unique subjects
        self.subset = subset
        self.aug_p = aug_p
        
        # Define MONAI transforms
        self.test_transforms = t.Compose([t.NormalizeIntensity()])
        self.monai_transforms = t.Compose([
            t.ToTensor(),
            t.Rand3DElastic(sigma_range=(0.01, 1), magnitude_range=(0, 1),
                             prob=aug_p, rotate_range=(0.18, 0.18, 0.18),
                             translate_range=(4, 4, 4), scale_range=(0.10, 0.10, 0.10),
                             spatial_size=None, padding_mode="border"),
            t.RandHistogramShift(num_control_points=(5, 15), prob=aug_p),
            t.RandAdjustContrast(prob=aug_p),
            t.RandGaussianNoise(prob=aug_p),
            t.NormalizeIntensity()
        ])
    
    def __len__(self):
        return len(self.subjects)
    
    def __getitem__(self, idx):
        subject_id = self.subjects[idx]
        subject_data = self.subject_groups.loc[subject_id]
        print(subject_data)
        
        # Load all images for this subject in sequence
        images = []
        for file_path in subject_data['File_Path']:
            img = nib.load(file_path).get_fdata()
            if img.ndim == 3:
                img = np.expand_dims(img, axis=0)  # Add channel dimension
            images.append(img)
        
        images = np.stack(images)  # Shape: (T, C, H, W, D), where T = sequence length
        
        # Apply transforms
        if self.subset == 'train' and self.aug_p > 0:
            images = self.monai_transforms(images)
        else:
            images = self.test_transforms(images)
        
        # Extract labels (taking the first available label for classification & regression)
        class_label = torch.tensor(subject_data.iloc[0]['Sex']).unsqueeze(0)
        reg_label = torch.tensor(subject_data.iloc[0]['Age']).unsqueeze(0)
        
        return images, class_label, reg_label

In [8]:
dataset = ADNISequenceDataset('/scratch/anirudhkaushik/ADNI1_Complete_Updated.csv', subset='train', aug_p=0.8)
dataloader = MultiEpochsDataLoader(dataset,
                                       batch_size=1,
                                       drop_last=False,
                                       shuffle=True,
                                       num_workers=4,
                                       pin_memory=True,
                                       prefetch_factor=1)

for i, (images, class_label, reg_label) in enumerate(dataloader):
    pass

    Image Data ID     Subject Group Sex  Age Visit Modality  \
660        I49675  073_S_0089    CN   M   65    sc      MRI   
661       I119227  073_S_0089    CN   M   65    sc      MRI   
662        I80648  073_S_0089    CN   M   66   m06      MRI   
665       I119228  073_S_0089    CN   M   66   m06      MRI   
663        I72472  073_S_0089    CN   M   66   m12      MRI   
664        I94601  073_S_0089    CN   M   67   m24      MRI   
659       I135801  073_S_0089    CN   M   68   m36      MRI   

                                      Description       Type   Acq Date  \
660    MPR-R; GradWarp; B1 Correction; N3; Scaled  Processed 2006-01-26   
661  MPR-R; GradWarp; B1 Correction; N3; Scaled_2  Processed 2006-01-26   
662      MPR; GradWarp; B1 Correction; N3; Scaled  Processed 2006-08-31   
665    MPR; GradWarp; B1 Correction; N3; Scaled_2  Processed 2006-08-31   
663      MPR; GradWarp; B1 Correction; N3; Scaled  Processed 2007-02-07   
664    MPR-R; GradWarp; B1 Correction; N3; Sc

RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home2/anirudhkaushik/miniconda3/envs/pytorch/lib/python3.10/site-packages/monai/transforms/transform.py", line 141, in apply_transform
    return _apply_transform(transform, data, unpack_items, lazy, overrides, log_stats)
  File "/home2/anirudhkaushik/miniconda3/envs/pytorch/lib/python3.10/site-packages/monai/transforms/transform.py", line 98, in _apply_transform
    return transform(data, lazy=lazy) if isinstance(transform, LazyTrait) else transform(data)
  File "/home2/anirudhkaushik/miniconda3/envs/pytorch/lib/python3.10/site-packages/monai/transforms/spatial/array.py", line 2905, in __call__
    out: torch.Tensor = self.resampler(
  File "/home2/anirudhkaushik/miniconda3/envs/pytorch/lib/python3.10/site-packages/monai/transforms/spatial/array.py", line 2108, in __call__
    grid_t[0, ..., i] *= 2.0 / max(2, dim)
IndexError: index 3 is out of bounds for dimension 2 